# SER Preprocessing — Kaggle version (T4 x2 GPU, prints every step)

Ported from your local Session1 notebook. What changed for Kaggle:

- Dataset is read from `/kaggle/input/...` (auto-detected, no hard-coded Windows path)
- Outputs are written to `/kaggle/working/`
- Resample, silence-trim, spectral-subtraction denoise, pre-emphasis and log-Mel Fbank run on the **GPU** (`cuda:0`)
- **Every preprocessing step is printed**: full step-by-step trace for the first few utterances, plus a one-line summary for every utterance

**Before running:** Notebook settings → Accelerator → **GPU T4 x2**, and click **Add Input** → add your IEMOCAP dataset. Then *Run All*.

## 1. Environment check (GPU, libraries)

In [1]:
import os, re, sys, time, glob
from dataclasses import dataclass
from typing import List, Dict, Tuple, Optional

import numpy as np
import pandas as pd
import soundfile as sf
import torch
import torch.nn.functional as F
import torchaudio
import torchaudio.transforms as T
from torch.utils.data import Dataset, DataLoader

print("Python      :", sys.version.split()[0])
print("PyTorch     :", torch.__version__)
print("Torchaudio  :", torchaudio.__version__)
print("CUDA avail  :", torch.cuda.is_available())
n_gpu = torch.cuda.device_count()
print("GPU count   :", n_gpu)
for i in range(n_gpu):
    p = torch.cuda.get_device_properties(i)
    print(f"  cuda:{i} -> {p.name}, {p.total_memory/1024**3:.1f} GB")

# Preprocessing is done on ONE GPU (cuda:0). It is mostly file-reading bound, so a
# second GPU gives no real speed-up here; T4 x2 becomes useful when you TRAIN the model.
DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)
if DEVICE.type == "cpu":
    print("[WARN] No GPU found. Turn on Settings -> Accelerator -> GPU T4 x2 and restart.")

Python      : 3.12.13
PyTorch     : 2.10.0+cpu
Torchaudio  : 2.10.0+cpu
CUDA avail  : False
GPU count   : 0
Using device: cpu
[WARN] No GPU found. Turn on Settings -> Accelerator -> GPU T4 x2 and restart.


## 2. Configuration
Only edit this cell if needed. `INPUT_ROOT` normally stays `/kaggle/input` — the notebook searches inside it for your IEMOCAP files.

In [2]:
INPUT_ROOT = "/kaggle/input/datasets/sangayb/iemocap"          # where Kaggle mounts your added datasets
WORK_DIR   = "/kaggle/working"        # anything saved here appears in the Output tab
OUT_DIR    = os.path.join(WORK_DIR, "processed")
MANIFEST_PATH = os.path.join(WORK_DIR, "manifest.csv")

SESSIONS      = [1]      # Session1 only. Use [1,2,3,4,5] for all sessions later.
NUM_SAMPLES   = None     # None = use ALL data in the manifest (no sampling)
SEED          = 42
DO_DENOISE    = True
DO_TRIM       = True

VERBOSE_FIRST_N = 3      # full step-by-step trace for the first N utterances
PRINT_EVERY_ROW = True   # one summary line for EVERY utterance (set False to silence)

os.makedirs(OUT_DIR, exist_ok=True)
print("Config OK ->", dict(SESSIONS=SESSIONS, NUM_SAMPLES=NUM_SAMPLES, DO_DENOISE=DO_DENOISE, DO_TRIM=DO_TRIM))

Config OK -> {'SESSIONS': [1], 'NUM_SAMPLES': None, 'DO_DENOISE': True, 'DO_TRIM': True}


## 3. Find your dataset inside `/kaggle/input`
Walks the input folder, finds every `EmoEvaluation` folder and indexes every `.wav` by utterance id. This works even if your dataset has a slightly different folder layout than the original IEMOCAP release.

In [3]:
print("Contents of", INPUT_ROOT, ":")
for d in sorted(os.listdir(INPUT_ROOT)):
    print("  ", d)

def discover_dataset(input_root: str):
    emo_dirs, wav_index = [], {}
    for dirpath, dirnames, filenames in os.walk(input_root):
        if os.path.basename(dirpath) == "EmoEvaluation":
            emo_dirs.append(dirpath)
        for f in filenames:
            if f.lower().endswith(".wav"):
                wav_index[os.path.splitext(f)[0]] = os.path.join(dirpath, f)
    return sorted(emo_dirs), wav_index

t0 = time.time()
EMO_DIRS, WAV_INDEX = discover_dataset(INPUT_ROOT)
print(f"\nScan finished in {time.time()-t0:.1f}s")
print("EmoEvaluation folders found:", len(EMO_DIRS))
for d in EMO_DIRS:
    print("   ", d)
print("Total .wav files indexed   :", len(WAV_INDEX))
print("Example wav                :", next(iter(WAV_INDEX.items())) if WAV_INDEX else None)

assert EMO_DIRS, "No 'EmoEvaluation' folder found - check that your IEMOCAP dataset was added via Add Input."
assert WAV_INDEX, "No .wav files found in /kaggle/input."

Contents of /kaggle/input/datasets/sangayb/iemocap :
   Session1
   Session2
   Session3
   Session4
   Session5

Scan finished in 7.2s
EmoEvaluation folders found: 5
    /kaggle/input/datasets/sangayb/iemocap/Session1/dialog/EmoEvaluation
    /kaggle/input/datasets/sangayb/iemocap/Session2/dialog/EmoEvaluation
    /kaggle/input/datasets/sangayb/iemocap/Session3/dialog/EmoEvaluation
    /kaggle/input/datasets/sangayb/iemocap/Session4/dialog/EmoEvaluation
    /kaggle/input/datasets/sangayb/iemocap/Session5/dialog/EmoEvaluation
Total .wav files indexed   : 10039
Example wav                : ('Ses04F_script01_2_M018', '/kaggle/input/datasets/sangayb/iemocap/Session4/sentences/wav/Ses04F_script01_2/Ses04F_script01_2_M018.wav')


## 4. Audio-cleaning functions (denoise + trim)
1. **NumPy reference versions** — exactly your original code (kept to verify the GPU versions).
2. **GPU versions** — same maths, vectorised in PyTorch, so no Python frame loops.

In [4]:
# ---------------- NumPy reference (your original code, unchanged) ----------------
def spectral_subtraction_denoise_np(wav, sr=16000, frame_ms=25.0, hop_ms=10.0,
                                    noise_percentile=10.0, over_subtraction=1.8,
                                    spectral_floor=0.05):
    frame_len = int(round(sr * frame_ms / 1000.0))
    hop_len = int(round(sr * hop_ms / 1000.0))
    window = np.hamming(frame_len)
    n = len(wav)
    n_frames = 1 + max(0, (n - frame_len) // hop_len)
    if n_frames <= 0:
        return wav
    spec = []
    for i in range(n_frames):
        start = i * hop_len
        frame = wav[start:start + frame_len]
        if len(frame) < frame_len:
            frame = np.pad(frame, (0, frame_len - len(frame)))
        spec.append(np.fft.rfft(frame * window))
    spec = np.stack(spec, axis=0)
    mag, phase = np.abs(spec), np.angle(spec)
    frame_energy = np.sum(mag ** 2, axis=1)
    thresh = np.percentile(frame_energy, noise_percentile)
    noise_frames = mag[frame_energy <= thresh]
    noise_spectrum = (np.mean(noise_frames, axis=0) if noise_frames.shape[0] > 0
                      else np.min(mag, axis=0))
    clean_mag = mag - over_subtraction * noise_spectrum[None, :]
    clean_mag = np.maximum(clean_mag, spectral_floor * mag)
    clean_spec = clean_mag * np.exp(1j * phase)
    out = np.zeros(n, dtype=np.float64)
    win_sum = np.zeros(n, dtype=np.float64)
    for i in range(n_frames):
        start = i * hop_len
        frame_time = np.fft.irfft(clean_spec[i], n=frame_len)
        end = min(start + frame_len, n)
        out[start:end] += (frame_time * window)[: end - start]
        win_sum[start:end] += (window ** 2)[: end - start]
    win_sum[win_sum < 1e-8] = 1e-8
    return (out / win_sum).astype(np.float32)


def trim_silence_np(wav, sr=16000, frame_ms=25.0, hop_ms=10.0, energy_threshold_ratio=0.02):
    frame_len = int(round(sr * frame_ms / 1000.0))
    hop_len = int(round(sr * hop_ms / 1000.0))
    n = len(wav)
    n_frames = 1 + max(0, (n - frame_len) // hop_len)
    if n_frames <= 1:
        return wav
    energies = np.array([
        np.sqrt(np.mean(wav[i * hop_len: i * hop_len + frame_len].astype(np.float64) ** 2) + 1e-12)
        for i in range(n_frames)
    ])
    thresh = energies.max() * energy_threshold_ratio
    active = np.where(energies > thresh)[0]
    if len(active) == 0:
        return wav
    start_sample = active[0] * hop_len
    end_sample = min(n, active[-1] * hop_len + frame_len)
    return wav[start_sample:end_sample]


# ---------------- GPU versions (same maths, vectorised) ----------------
@torch.no_grad()
def trim_silence_gpu(wav: torch.Tensor, sr: int = 16000, frame_ms: float = 25.0,
                     hop_ms: float = 10.0, energy_threshold_ratio: float = 0.02) -> torch.Tensor:
    frame_len = int(round(sr * frame_ms / 1000.0))
    hop_len = int(round(sr * hop_ms / 1000.0))
    n = wav.numel()
    if n < frame_len:
        return wav
    frames = wav.unfold(0, frame_len, hop_len)                 # (n_frames, frame_len)
    if frames.shape[0] <= 1:
        return wav
    energies = torch.sqrt(frames.double().pow(2).mean(dim=1) + 1e-12)
    thresh = energies.max() * energy_threshold_ratio
    active = torch.nonzero(energies > thresh).flatten()
    if active.numel() == 0:
        return wav
    start_sample = int(active[0].item()) * hop_len
    end_sample = min(n, int(active[-1].item()) * hop_len + frame_len)
    return wav[start_sample:end_sample]


@torch.no_grad()
def spectral_subtraction_denoise_gpu(wav: torch.Tensor, sr: int = 16000, frame_ms: float = 25.0,
                                     hop_ms: float = 10.0, noise_percentile: float = 10.0,
                                     over_subtraction: float = 1.8,
                                     spectral_floor: float = 0.05) -> torch.Tensor:
    frame_len = int(round(sr * frame_ms / 1000.0))
    hop_len = int(round(sr * hop_ms / 1000.0))
    n = wav.numel()
    x = wav
    if n < frame_len:                                          # single zero-padded frame
        x = F.pad(x, (0, frame_len - n))
    window = torch.hamming_window(frame_len, periodic=False, device=wav.device, dtype=wav.dtype)

    frames = x.unfold(0, frame_len, hop_len)                   # (n_frames, frame_len)
    n_frames = frames.shape[0]
    spec = torch.fft.rfft(frames * window, dim=1)              # (n_frames, n_bins)
    mag, phase = spec.abs(), spec.angle()

    frame_energy = (mag ** 2).sum(dim=1)
    thresh = torch.quantile(frame_energy, noise_percentile / 100.0)
    noise_frames = mag[frame_energy <= thresh]
    noise_spectrum = noise_frames.mean(dim=0) if noise_frames.shape[0] > 0 else mag.min(dim=0).values

    clean_mag = torch.maximum(mag - over_subtraction * noise_spectrum[None, :], spectral_floor * mag)
    clean_spec = torch.polar(clean_mag, phase)
    frame_time = torch.fft.irfft(clean_spec, n=frame_len, dim=1) * window   # (n_frames, frame_len)

    total_len = (n_frames - 1) * hop_len + frame_len
    out = F.fold(frame_time.t().unsqueeze(0), output_size=(1, total_len),
                 kernel_size=(1, frame_len), stride=(1, hop_len)).view(-1)
    wsum = F.fold((window ** 2).unsqueeze(1).expand(frame_len, n_frames).unsqueeze(0).contiguous(),
                  output_size=(1, total_len), kernel_size=(1, frame_len),
                  stride=(1, hop_len)).view(-1)

    out, wsum = out[:n], wsum[:n]                              # crop to original length
    if out.numel() < n:                                        # samples past the last frame stay 0
        out = F.pad(out, (0, n - out.numel()))
        wsum = F.pad(wsum, (0, n - wsum.numel()))
    wsum = torch.clamp(wsum, min=1e-8)
    return out / wsum

print("Cleaning functions defined (NumPy reference + GPU).")

Cleaning functions defined (NumPy reference + GPU).


## 5. IEMOCAP manifest builder
Reads the real annotator votes from `EmoEvaluation/*.txt` (6 classes; excitement merged into happiness). Wav files are looked up in the index built in step 3, so it works with any folder layout.

In [5]:
VOTE_COLUMNS = ["votes_anger", "votes_happy", "votes_sad",
                "votes_neutral", "votes_frustration", "votes_fear"]

CATEGORY_MAP = {
    "anger": "votes_anger",
    "happiness": "votes_happy",
    "excitement": "votes_happy",      # merged into happiness
    "sadness": "votes_sad",
    "neutral": "votes_neutral",
    "frustration": "votes_frustration",
    "fear": "votes_fear",
}

UTTERANCE_HEADER_RE = re.compile(r"^\[\s*[\d.]+\s*-\s*[\d.]+\s*\]\s+(\S+)\s+(\S+)\s+\[.*\]\s*$")
CATEGORICAL_LINE_RE = re.compile(r"^C-\w+:\s*(.+?)\s*\(.*\)\s*$")


def parse_emo_evaluation_file(txt_path: str) -> Dict[str, Dict[str, int]]:
    results: Dict[str, Dict[str, int]] = {}
    current_utt_id: Optional[str] = None
    with open(txt_path, "r", encoding="utf-8", errors="ignore") as f:
        for raw_line in f:
            line = raw_line.rstrip("\n")
            m = UTTERANCE_HEADER_RE.match(line)
            if m:
                current_utt_id = m.group(1)
                results[current_utt_id] = {col: 0 for col in VOTE_COLUMNS}
                continue
            if current_utt_id is None:
                continue
            c = CATEGORICAL_LINE_RE.match(line)
            if c:
                labels = [l.strip().lower() for l in c.group(1).split(";") if l.strip()]
                seen = set()
                for lbl in labels:
                    col = CATEGORY_MAP.get(lbl)
                    if col and col not in seen:
                        results[current_utt_id][col] += 1
                        seen.add(col)
    return results


def session_num_for_utterance(utt_id: str) -> int:
    m = re.match(r"Ses(\d{2})", utt_id)
    return int(m.group(1)) if m else 0


def speaker_for_utterance(session_num: int, utt_id: str) -> str:
    last = utt_id.split("_")[-1]
    gender = last[0].upper() if last and last[0].upper() in ("F", "M") else "?"
    return f"Ses{session_num:02d}{gender}"


def build_manifest(emo_dirs, wav_index, sessions=(1,), drop_zero_vote_rows=True) -> pd.DataFrame:
    rows, missing_wav = [], 0
    for emo_dir in emo_dirs:
        txt_files = sorted(f for f in os.listdir(emo_dir)
                           if f.endswith(".txt") and os.path.isfile(os.path.join(emo_dir, f)))
        for txt_name in txt_files:
            txt_path = os.path.join(emo_dir, txt_name)
            try:
                utt_votes = parse_emo_evaluation_file(txt_path)
            except Exception as e:
                print(f"[WARN] Failed to parse {txt_path}: {e}")
                continue
            for utt_id, votes in utt_votes.items():
                s = session_num_for_utterance(utt_id)
                if s not in sessions:
                    continue
                if drop_zero_vote_rows and sum(votes.values()) == 0:
                    continue
                wav_path = wav_index.get(utt_id)
                if wav_path is None:
                    missing_wav += 1
                    continue
                row = {"file_path": wav_path, "session": f"Session{s}",
                       "speaker": speaker_for_utterance(s, utt_id), "utt_name": utt_id}
                row.update({c: votes[c] for c in VOTE_COLUMNS})
                rows.append(row)
    if missing_wav:
        print(f"[WARN] {missing_wav} utterances had votes but no matching .wav in the dataset (skipped)")
    return pd.DataFrame(rows, columns=["file_path", "session", "speaker", "utt_name"] + VOTE_COLUMNS)

print("Manifest builder defined.")

Manifest builder defined.


## 6. Build `manifest.csv`

In [6]:
df = build_manifest(EMO_DIRS, WAV_INDEX, sessions=SESSIONS, drop_zero_vote_rows=True)
df.to_csv(MANIFEST_PATH, index=False)

print(f"Wrote {len(df)} utterance rows to {MANIFEST_PATH}")
print("Sessions found:", sorted(df["session"].unique().tolist()))
print("Speakers found:", sorted(df["speaker"].unique().tolist()))
print("\nVote totals per class:")
print(df[VOTE_COLUMNS].sum())
df.head(10)

Wrote 1799 utterance rows to /kaggle/working/manifest.csv
Sessions found: ['Session1']
Speakers found: ['Ses01F', 'Ses01M']

Vote totals per class:
votes_anger          1026
votes_happy           772
votes_sad             827
votes_neutral        1527
votes_frustration    1285
votes_fear             99
dtype: int64


,file_path,session,speaker,utt_name,votes_anger,votes_happy,votes_sad,votes_neutral,votes_frustration,votes_fear
0,/kaggle/input/datasets/sangayb/iemocap/Session...,Session1,Ses01F,Ses01F_impro01_F000,0,0,0,4,0,0
1,/kaggle/input/datasets/sangayb/iemocap/Session...,Session1,Ses01F,Ses01F_impro01_F001,1,0,0,4,0,0
2,/kaggle/input/datasets/sangayb/iemocap/Session...,Session1,Ses01F,Ses01F_impro01_F002,1,0,0,3,0,0
3,/kaggle/input/datasets/sangayb/iemocap/Session...,Session1,Ses01F,Ses01F_impro01_F003,2,0,0,1,2,0
4,/kaggle/input/datasets/sangayb/iemocap/Session...,Session1,Ses01F,Ses01F_impro01_F004,2,0,0,1,2,0
5,/kaggle/input/datasets/sangayb/iemocap/Session...,Session1,Ses01F,Ses01F_impro01_F005,1,0,0,3,1,0
6,/kaggle/input/datasets/sangayb/iemocap/Session...,Session1,Ses01F,Ses01F_impro01_F006,1,0,0,1,3,0
7,/kaggle/input/datasets/sangayb/iemocap/Session...,Session1,Ses01F,Ses01F_impro01_F007,2,0,0,0,4,0
8,/kaggle/input/datasets/sangayb/iemocap/Session...,Session1,Ses01F,Ses01F_impro01_F008,2,0,0,0,3,0
9,/kaggle/input/datasets/sangayb/iemocap/Session...,Session1,Ses01F,Ses01F_impro01_F009,2,0,0,0,4,0


## 7. Constants, manifest loader and label construction (Eq. 1–5)

In [7]:
TARGET_SR = 16000
PREEMPH_COEF = 0.97
MAX_SEG_SEC = 7.0
MIN_SEG_SEC = 1.5
N_MELS = 40
WIN_MS = 40.0
HOP_MS = 10.0

EMOTION_CLASSES = ["anger", "happiness", "sadness", "neutral", "frustration", "fear"]
LABEL2IDX = {e: i for i, e in enumerate(EMOTION_CLASSES)}
K = len(EMOTION_CLASSES)

VOTE_COL_FOR_CLASS = {
    "anger": "votes_anger", "happiness": "votes_happy", "sadness": "votes_sad",
    "neutral": "votes_neutral", "frustration": "votes_frustration", "fear": "votes_fear",
}
VOTE_COLS_IN_ORDER = [VOTE_COL_FOR_CLASS[e] for e in EMOTION_CLASSES]
REQUIRED_COLS = ["file_path"] + VOTE_COLS_IN_ORDER


def load_manifest(manifest_path, num_samples=None, shuffle=True, seed=42) -> pd.DataFrame:
    df = pd.read_csv(manifest_path)
    df.columns = [str(c).strip() for c in df.columns]
    missing = set(REQUIRED_COLS) - set(df.columns)
    if missing:
        raise ValueError(f"Manifest missing columns {missing}; found {list(df.columns)}")

    if num_samples is not None and num_samples < len(df):
        if not shuffle:
            df = df.iloc[:num_samples].reset_index(drop=True)
        elif "session" in df.columns:
            parts = []
            for _, g in df.groupby("session"):
                k = max(1, round(len(g) * num_samples / len(df)))
                parts.append(g.sample(n=min(k, len(g)), random_state=seed))
            df = pd.concat(parts)
            if len(df) > num_samples:
                df = df.sample(n=num_samples, random_state=seed)
            df = df.sample(frac=1.0, random_state=seed).reset_index(drop=True)
        else:
            df = df.sample(n=num_samples, random_state=seed).reset_index(drop=True)
    elif shuffle:
        df = df.sample(frac=1.0, random_state=seed).reset_index(drop=True)
    return df


def compute_labels(vote_row: Dict[str, int]):
    votes = np.array([vote_row[c] for c in VOTE_COLS_IN_ORDER], dtype=np.float64)
    total = votes.sum()
    if total <= 0:
        raise ValueError("Utterance has zero annotator votes; cannot label it.")
    soft_label = votes / total                                   # Eq. (5)
    top_idx = int(np.argmax(votes))
    is_clear = bool(votes[top_idx] > total / 2.0 and np.sum(votes == votes[top_idx]) == 1)
    hard_label = np.zeros(K, dtype=np.float32)
    if is_clear:
        hard_label[top_idx] = 1.0                                # Eq. (1)
    return ("SA" if is_clear else "SB"), hard_label, soft_label.astype(np.float32), (not is_clear)

print("Emotion classes (order = label-vector order):", EMOTION_CLASSES)

Emotion classes (order = label-vector order): ['anger', 'happiness', 'sadness', 'neutral', 'frustration', 'fear']


## 8. Waveform + Fbank functions (GPU)

In [8]:
_RESAMPLERS = {}

def get_resampler(orig_sr: int) -> T.Resample:
    if orig_sr not in _RESAMPLERS:
        _RESAMPLERS[orig_sr] = T.Resample(orig_freq=orig_sr, new_freq=TARGET_SR).to(DEVICE)
    return _RESAMPLERS[orig_sr]


def wav_stats(name: str, w: torch.Tensor) -> str:
    w = w.detach()
    rms = torch.sqrt(torch.mean(w.float() ** 2)).item() if w.numel() else 0.0
    return (f"[{name}] samples={w.numel():>7d}  dur={w.numel()/TARGET_SR:6.3f}s  "
            f"min={w.min().item():+.4f}  max={w.max().item():+.4f}  rms={rms:.5f}")


@torch.no_grad()
def apply_preemphasis(wav: torch.Tensor, coef: float = PREEMPH_COEF) -> torch.Tensor:
    if wav.numel() < 2:
        return wav
    out = torch.empty_like(wav)
    out[0] = wav[0]
    out[1:] = wav[1:] - coef * wav[:-1]
    return out


def segment_and_pad(wav: torch.Tensor, sr: int = TARGET_SR,
                    max_sec: float = MAX_SEG_SEC, min_sec: float = MIN_SEG_SEC) -> List[torch.Tensor]:
    max_len, min_len = int(round(max_sec * sr)), int(round(min_sec * sr))
    n = wav.numel()
    chunks = []
    if n > max_len:
        start = 0
        while start < n:
            end = min(start + max_len, n)
            chunks.append(wav[start:end])
            start = end
    else:
        chunks.append(wav)
    out = []
    for c in chunks:
        if c.numel() < min_len:
            c = F.pad(c, (0, min_len - c.numel()), mode="constant", value=0.0)   # pad at END
        out.append(c)
    return out


def make_fbank_extractor(sr=TARGET_SR, n_mels=N_MELS, win_ms=WIN_MS, hop_ms=HOP_MS):
    win_length = int(round(sr * win_ms / 1000.0))
    hop_length = int(round(sr * hop_ms / 1000.0))
    n_fft = int(2 ** np.ceil(np.log2(win_length)))
    return T.MelSpectrogram(sample_rate=sr, n_fft=n_fft, win_length=win_length, hop_length=hop_length,
                            window_fn=torch.hamming_window, n_mels=n_mels, power=2.0).to(DEVICE)


@torch.no_grad()
def compute_log_fbank(wav: torch.Tensor, mel_extractor, eps: float = 1e-6) -> torch.Tensor:
    mel = mel_extractor(wav.unsqueeze(0))       # (1, M, f)
    return torch.log(mel + eps).transpose(1, 2) # (1, f, M)

print("Waveform / Fbank functions defined.")

Waveform / Fbank functions defined.


## 9. Verify GPU cleaning matches your original NumPy code
Runs both versions on one real utterance and prints the difference (should be tiny).

In [9]:
test_path = df["file_path"].iloc[0]
d, sr0 = sf.read(test_path, dtype="float32", always_2d=True)
w = torch.from_numpy(d.T).mean(dim=0)
if sr0 != TARGET_SR:
    w = T.Resample(sr0, TARGET_SR)(w.unsqueeze(0)).squeeze(0)
w_np = w.numpy()

t_np = trim_silence_np(w_np, TARGET_SR)
t_gp = trim_silence_gpu(w.to(DEVICE), TARGET_SR).cpu().numpy()
print("TRIM    -> numpy len:", len(t_np), " gpu len:", len(t_gp), " same length:", len(t_np) == len(t_gp))

dn_np = spectral_subtraction_denoise_np(t_np, TARGET_SR)
dn_gp = spectral_subtraction_denoise_gpu(torch.from_numpy(t_np).to(DEVICE), TARGET_SR).cpu().numpy()
print("DENOISE -> numpy len:", len(dn_np), " gpu len:", len(dn_gp))
print("           max abs diff :", float(np.max(np.abs(dn_np - dn_gp))))
print("           mean abs diff:", float(np.mean(np.abs(dn_np - dn_gp))))
print("           signal max amp:", float(np.max(np.abs(dn_np))))

TRIM    -> numpy len: 31120  gpu len: 31120  same length: True
DENOISE -> numpy len: 31120  gpu len: 31120
           max abs diff : 1.4901161193847656e-08
           mean abs diff: 5.5717602753802e-10
           signal max amp: 0.08695333451032639


## 10. Per-utterance pipeline (with step-by-step printing)

In [10]:
@dataclass
class ProcessedSample:
    utt_id: str
    split: str
    waveform: torch.Tensor
    fbank: torch.Tensor
    hard_label: torch.Tensor
    soft_label: torch.Tensor
    is_ambiguous: bool
    session: str = ""
    speaker: str = ""


@torch.no_grad()
def process_one_utterance(row: pd.Series, mel_extractor, utt_id: str,
                          do_denoise: bool = True, do_trim: bool = True,
                          verbose: bool = False) -> List[ProcessedSample]:
    log = print if verbose else (lambda *a, **k: None)
    log("=" * 100)
    log(f"UTTERANCE {utt_id}   file = {row['file_path']}")
    log("=" * 100)

    # STEP 1: labels
    split, hard_label, soft_label, is_ambig = compute_labels(row.to_dict())
    log("STEP 1  Labels from annotator votes")
    log("        votes      :", {c.replace('votes_', ''): int(row[c]) for c in VOTE_COLS_IN_ORDER})
    log("        soft label :", np.round(soft_label, 3).tolist(), " (order:", EMOTION_CLASSES, ")")
    log("        split      :", split, "(clear)" if split == "SA" else "(ambiguous)")
    log("        hard label :", hard_label.tolist())

    # STEP 2: load raw audio
    data, sr = sf.read(row["file_path"], dtype="float32", always_2d=True)
    log(f"STEP 2  Load raw audio        : sample_rate={sr} Hz, channels={data.shape[1]}, samples={data.shape[0]}")
    wav = torch.from_numpy(data.T).to(DEVICE)
    if wav.shape[0] > 1:
        wav = wav.mean(dim=0, keepdim=True)
        log("        downmixed to mono")
    wav = wav.squeeze(0)
    raw_len = wav.numel()
    log("        ", wav_stats("raw", wav))

    # STEP 3: resample
    if sr != TARGET_SR:
        wav = get_resampler(sr)(wav.unsqueeze(0)).squeeze(0)
        log(f"STEP 3  Resample {sr} -> {TARGET_SR} Hz")
    else:
        log(f"STEP 3  Resample: already {TARGET_SR} Hz, nothing to do")
    log("        ", wav_stats("resampled", wav))

    # STEP 4: trim silence
    if do_trim:
        before = wav.numel()
        wav = trim_silence_gpu(wav, sr=TARGET_SR)
        log(f"STEP 4  Trim silence          : {before} -> {wav.numel()} samples "
            f"({(before-wav.numel())/TARGET_SR:.3f}s removed)")
    else:
        log("STEP 4  Trim silence          : SKIPPED")
    log("        ", wav_stats("trimmed", wav))

    # STEP 5: denoise
    if do_denoise:
        rms_before = torch.sqrt(torch.mean(wav ** 2)).item()
        wav = spectral_subtraction_denoise_gpu(wav, sr=TARGET_SR)
        rms_after = torch.sqrt(torch.mean(wav ** 2)).item()
        log(f"STEP 5  Spectral-subtraction denoise : RMS {rms_before:.5f} -> {rms_after:.5f}")
    else:
        log("STEP 5  Denoise                : SKIPPED")
    log("        ", wav_stats("denoised", wav))

    # STEP 6: pre-emphasis
    wav = apply_preemphasis(wav)
    log(f"STEP 6  Pre-emphasis (coef={PREEMPH_COEF}) : y[t] = x[t] - {PREEMPH_COEF} * x[t-1]")
    log("        ", wav_stats("pre-emph", wav))

    # STEP 7: segment / pad
    chunks = segment_and_pad(wav)
    log(f"STEP 7  Segment (>{MAX_SEG_SEC}s) / zero-pad end (<{MIN_SEG_SEC}s) : {len(chunks)} chunk(s), "
        f"lengths = {[c.numel() for c in chunks]}")

    samples = []
    for i, chunk in enumerate(chunks):
        # STEP 8: log-Mel Fbank
        fbank = compute_log_fbank(chunk, mel_extractor)
        log(f"STEP 8  Log-Mel Fbank (chunk {i}) : shape={tuple(fbank.shape)}  (1, frames, {N_MELS} mels)  "
            f"min={fbank.min().item():.3f} max={fbank.max().item():.3f} mean={fbank.mean().item():.3f}")
        chunk_id = utt_id if len(chunks) == 1 else f"{utt_id}_seg{i}"
        samples.append(ProcessedSample(
            utt_id=chunk_id, split=split,
            waveform=chunk.float().cpu(), fbank=fbank.float().cpu(),
            hard_label=torch.from_numpy(hard_label), soft_label=torch.from_numpy(soft_label),
            is_ambiguous=is_ambig,
            session=str(row.get("session", "")), speaker=str(row.get("speaker", "")),
        ))
    log(f"DONE    raw {raw_len/sr:.3f}s -> final {[round(c.numel()/TARGET_SR, 3) for c in chunks]} s\n")
    return samples

print("process_one_utterance defined.")

process_one_utterance defined.


## 11. Dataset + collate function

In [11]:
class SERDataset(Dataset):
    def __init__(self, samples: List[ProcessedSample]):
        self.samples = samples
    def __len__(self):
        return len(self.samples)
    def __getitem__(self, idx):
        s = self.samples[idx]
        label = s.hard_label if s.split == "SA" else s.soft_label
        return {"utt_id": s.utt_id, "waveform": s.waveform, "fbank": s.fbank,
                "label": label, "split": s.split, "is_ambiguous": s.is_ambiguous}


def collate_fn(batch):
    max_wav_len = max(b["waveform"].shape[-1] for b in batch)
    max_frames = max(b["fbank"].shape[1] for b in batch)
    waveforms, fbanks, labels, splits, utt_ids, is_ambig = [], [], [], [], [], []
    for b in batch:
        waveforms.append(F.pad(b["waveform"], (0, max_wav_len - b["waveform"].shape[-1])))
        fbanks.append(F.pad(b["fbank"], (0, 0, 0, max_frames - b["fbank"].shape[1])))
        labels.append(b["label"]); splits.append(b["split"])
        utt_ids.append(b["utt_id"]); is_ambig.append(b["is_ambiguous"])
    return {"utt_id": utt_ids, "waveform": torch.stack(waveforms), "fbank": torch.stack(fbanks),
            "label": torch.stack(labels), "split": splits,
            "is_ambiguous": torch.tensor(is_ambig, dtype=torch.bool)}


def load_processed(cache_path: str) -> SERDataset:
    blob = torch.load(cache_path, weights_only=False)
    return SERDataset([ProcessedSample(**d) for d in blob["samples"]])

print("Dataset / collate defined.")

Dataset / collate defined.


## 12. Run preprocessing
First 3 utterances get the **full step-by-step trace**; every utterance then gets a one-line summary.

In [12]:
def run_preprocessing(manifest_path, out_dir, num_samples=1000, shuffle=True, seed=42,
                      do_denoise=True, do_trim=True,
                      verbose_first_n=3, print_every_row=True) -> str:
    os.makedirs(out_dir, exist_ok=True)
    df_run = load_manifest(manifest_path, num_samples=num_samples, shuffle=shuffle, seed=seed)
    print(f"[INFO] Manifest rows selected: {len(df_run)}")
    print("[INFO] Rows per speaker:", df_run["speaker"].value_counts().to_dict() if "speaker" in df_run else "n/a")
    mel_extractor = make_fbank_extractor()

    all_samples, n_sa, n_sb, n_fail = [], 0, 0, 0
    sessions_seen = set()
    t0 = time.time()

    for i, row in df_run.iterrows():
        utt_id = f"utt_{i:06d}"
        try:
            processed = process_one_utterance(row, mel_extractor, utt_id, do_denoise=do_denoise,
                                              do_trim=do_trim, verbose=(i < verbose_first_n))
        except Exception as e:
            n_fail += 1
            print(f"[WARN] Skipping row {i} ({row.get('file_path')}): {e}")
            continue

        for p in processed:
            n_sa += p.split == "SA"
            n_sb += p.split == "SB"
            sessions_seen.add(p.session)
        all_samples.extend(processed)

        if print_every_row and i >= verbose_first_n:
            p0 = processed[0]
            print(f"[{i+1:>4}/{len(df_run)}] {utt_id} | {row.get('utt_name', '')} | {p0.split} | "
                  f"segs={len(processed)} | wave={tuple(p0.waveform.shape)} | fbank={tuple(p0.fbank.shape)}")

    tag = "all" if num_samples is None else str(num_samples)
    out_path = os.path.join(out_dir, f"processed_first_{tag}.pt")
    torch.save({
        "samples": [s.__dict__ for s in all_samples],
        "meta": {
            "num_source_rows": len(df_run), "num_processed_segments": len(all_samples),
            "num_SA_clear": n_sa, "num_SB_ambiguous": n_sb,
            "sessions_represented": sorted(sessions_seen), "emotion_classes": EMOTION_CLASSES,
            "target_sr": TARGET_SR, "n_mels": N_MELS, "win_ms": WIN_MS, "hop_ms": HOP_MS,
            "denoised": do_denoise, "trimmed": do_trim,
        }
    }, out_path)

    print("\n" + "=" * 70)
    print(f"[INFO] Time taken            : {time.time()-t0:.1f}s")
    print(f"[INFO] Manifest rows -> segs : {len(df_run)} -> {len(all_samples)}  (failed rows: {n_fail})")
    print(f"[INFO] SA (clear) = {n_sa}   SB (ambiguous) = {n_sb}")
    print(f"[INFO] Sessions represented  : {sorted(sessions_seen)}")
    print(f"[INFO] Saved to              : {out_path}  ({os.path.getsize(out_path)/1024**2:.1f} MB)")
    return out_path


cache_path = run_preprocessing(
    manifest_path=MANIFEST_PATH, out_dir=OUT_DIR, num_samples=NUM_SAMPLES,
    shuffle=True, seed=SEED, do_denoise=DO_DENOISE, do_trim=DO_TRIM,
    verbose_first_n=VERBOSE_FIRST_N, print_every_row=PRINT_EVERY_ROW,
)

[INFO] Manifest rows selected: 1799
[INFO] Rows per speaker: {'Ses01M': 929, 'Ses01F': 870}
UTTERANCE utt_000000   file = /kaggle/input/datasets/sangayb/iemocap/Session1/sentences/wav/Ses01M_impro04/Ses01M_impro04_F005.wav
STEP 1  Labels from annotator votes
        votes      : {'anger': 0, 'happy': 0, 'sad': 0, 'neutral': 3, 'frustration': 1, 'fear': 0}
        soft label : [0.0, 0.0, 0.0, 0.75, 0.25, 0.0]  (order: ['anger', 'happiness', 'sadness', 'neutral', 'frustration', 'fear'] )
        split      : SA (clear)
        hard label : [0.0, 0.0, 0.0, 1.0, 0.0, 0.0]
STEP 2  Load raw audio        : sample_rate=16000 Hz, channels=1, samples=52800
         [raw] samples=  52800  dur= 3.300s  min=-0.1824  max=+0.1649  rms=0.02978
STEP 3  Resample: already 16000 Hz, nothing to do
         [resampled] samples=  52800  dur= 3.300s  min=-0.1824  max=+0.1649  rms=0.02978
STEP 4  Trim silence          : 52800 -> 52720 samples (0.005s removed)
         [trimmed] samples=  52720  dur= 3.295s  mi

## 13. Sanity check (shapes, label width = 6, NaN check)

In [13]:
blob = torch.load(cache_path, weights_only=False)
print("=== META ===")
for k, v in blob["meta"].items():
    print(f"  {k}: {v}")

dataset = load_processed(cache_path)
loader = DataLoader(dataset, batch_size=8, shuffle=True, collate_fn=collate_fn)
batch = next(iter(loader))
print("\n=== BATCH SHAPES ===")
print("  waveform:", tuple(batch["waveform"].shape))
print("  fbank   :", tuple(batch["fbank"].shape))
print("  label   :", tuple(batch["label"].shape), " <- (8, 6)")
print("  splits  :", batch["split"])

print("\n=== NaN / amplitude check on the whole dataset ===")
bad_wave = sum(torch.isnan(s.waveform).any().item() for s in dataset.samples)
bad_fb = sum(torch.isnan(s.fbank).any().item() for s in dataset.samples)
print("  segments with NaN waveform:", bad_wave)
print("  segments with NaN fbank   :", bad_fb)
print("  max |waveform| overall    :", max(s.waveform.abs().max().item() for s in dataset.samples))

=== META ===
  num_source_rows: 1799
  num_processed_segments: 2110
  num_SA_clear: 1649
  num_SB_ambiguous: 461
  sessions_represented: ['Session1']
  emotion_classes: ['anger', 'happiness', 'sadness', 'neutral', 'frustration', 'fear']
  target_sr: 16000
  n_mels: 40
  win_ms: 40.0
  hop_ms: 10.0
  denoised: True
  trimmed: True

=== BATCH SHAPES ===
  waveform: (8, 104080)
  fbank   : (8, 1, 651, 40)
  label   : (8, 6)  <- (8, 6)
  splits  : ['SA', 'SA', 'SA', 'SA', 'SB', 'SB', 'SA', 'SA']

=== NaN / amplitude check on the whole dataset ===
  segments with NaN waveform: 0
  segments with NaN fbank   : 0
  max |waveform| overall    : 1.7143950462341309


## 14. Fbank dimensions of every processed segment

In [14]:
all_samples = torch.load(cache_path, weights_only=False)["samples"]
print("=" * 64)
print(f"ORIGINAL DIMENSIONS FOR ALL {len(all_samples)} SPEECH SEGMENTS")
print("=" * 64)
for sample in all_samples:
    rows, cols = sample["fbank"].squeeze().shape
    print(f"({sample['utt_id']}) : {rows} rows and {cols} columns")

ORIGINAL DIMENSIONS FOR ALL 2110 SPEECH SEGMENTS
(utt_000000) : 330 rows and 40 columns
(utt_000001) : 158 rows and 40 columns
(utt_000002) : 338 rows and 40 columns
(utt_000003_seg0) : 701 rows and 40 columns
(utt_000003_seg1) : 151 rows and 40 columns
(utt_000004) : 323 rows and 40 columns
(utt_000005_seg0) : 701 rows and 40 columns
(utt_000005_seg1) : 151 rows and 40 columns
(utt_000006) : 168 rows and 40 columns
(utt_000007) : 423 rows and 40 columns
(utt_000008) : 151 rows and 40 columns
(utt_000009) : 224 rows and 40 columns
(utt_000010) : 260 rows and 40 columns
(utt_000011) : 183 rows and 40 columns
(utt_000012_seg0) : 701 rows and 40 columns
(utt_000012_seg1) : 194 rows and 40 columns
(utt_000013_seg0) : 701 rows and 40 columns
(utt_000013_seg1) : 313 rows and 40 columns
(utt_000014) : 387 rows and 40 columns
(utt_000015) : 272 rows and 40 columns
(utt_000016_seg0) : 701 rows and 40 columns
(utt_000016_seg1) : 611 rows and 40 columns
(utt_000017_seg0) : 701 rows and 40 columns

## 15. Live / own-recording preprocessing (same pipeline, for inference)
Upload your recording as a Kaggle dataset (or to `/kaggle/working`) and call `process_live_file(...)`.

In [15]:
@dataclass
class LiveProcessedSample:
    utt_id: str
    waveform: torch.Tensor
    fbank: torch.Tensor
    source_file: str


@torch.no_grad()
def process_live_file(file_path: str, mel_extractor, utt_id: str,
                      do_denoise: bool = True, do_trim: bool = True,
                      verbose: bool = True) -> List[LiveProcessedSample]:
    log = print if verbose else (lambda *a, **k: None)
    data, sr = sf.read(file_path, dtype="float32", always_2d=True)
    wav = torch.from_numpy(data.T).to(DEVICE).mean(dim=0)
    log(f"STEP 1  Load     : sr={sr}, samples={wav.numel()}")
    if sr != TARGET_SR:
        wav = get_resampler(sr)(wav.unsqueeze(0)).squeeze(0)
    log("STEP 2  Resample:", wav_stats("16k", wav))
    if do_trim:
        wav = trim_silence_gpu(wav, TARGET_SR)
    log("STEP 3  Trim    :", wav_stats("trim", wav))
    if do_denoise:
        wav = spectral_subtraction_denoise_gpu(wav, TARGET_SR)
    log("STEP 4  Denoise :", wav_stats("denoise", wav))
    wav = apply_preemphasis(wav)
    log("STEP 5  Pre-emph:", wav_stats("pre-emph", wav))
    chunks = segment_and_pad(wav)
    log(f"STEP 6  Segment  : {len(chunks)} chunk(s), lengths={[c.numel() for c in chunks]}")
    out = []
    for i, chunk in enumerate(chunks):
        fbank = compute_log_fbank(chunk, mel_extractor)
        log(f"STEP 7  Fbank    : chunk {i} -> {tuple(fbank.shape)}")
        cid = utt_id if len(chunks) == 1 else f"{utt_id}_seg{i}"
        out.append(LiveProcessedSample(cid, chunk.float().cpu(), fbank.float().cpu(), file_path))
    return out

# Example (uncomment and edit the path):
# live = process_live_file("/kaggle/input/my-recordings/test.wav", make_fbank_extractor(), "live_001")
print("process_live_file defined.")

process_live_file defined.


## 16. Files you can download / reuse

In [16]:
for root, _, files in os.walk(WORK_DIR):
    for f in files:
        p = os.path.join(root, f)
        if f.endswith((".csv", ".pt")):
            print(f"{os.path.getsize(p)/1024**2:8.2f} MB  {p}")
print("\nTo reuse in your training notebook: Save Version -> then Add Input -> 'Notebook output' of this notebook.")

    0.26 MB  /kaggle/working/manifest.csv
  622.32 MB  /kaggle/working/processed/processed_first_all.pt

To reuse in your training notebook: Save Version -> then Add Input -> 'Notebook output' of this notebook.
